# Public-transport weekday stop frequencies

This notebook combines the Verbund school-day and holiday weekday extracts with the supplied stop-location workbook. It keeps the preparation deliberately transparent: inspect the source counts, document the 2016 problem, build the usable years, and write the annual GeoParquets.

The original 2016 school-day export is incomplete. It contains 2,542 stops and 122,746 departures, while the 2017 school-day export contains 7,519 stops and 247,073 departures. Datei 3 contains exactly the same limited records as Datei 1, so this is a source-data problem rather than a parser or coordinate-matching problem. The analysis therefore uses the complete 2017 stop-frequency layer as the explicit proxy for 2016.


In [23]:
import os
from pathlib import Path

os.chdir(r"D:\CO2_Masterarbeit\CO2_Masterarbeit")

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("OGD/Public_Transport/Verbund_Daten_work")
OUTPUT_DIR = Path("OGD/Public_Transport")
SOURCE_YEARS = (2016, 2017, 2018, 2019, 2020, 2022)
VALID_SOURCE_YEARS = (2017, 2018, 2019, 2020, 2022)
EFA_PREFIX = 63_200_000


In [24]:
def read_frequency_csv(path):
    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype={"EFA-Haltestellennummer": "string"})
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path}")


def source_file(year, day_type, table_number=1):
    pattern = f"Manfred_Brandl_stv-{year}*/**/Montag-Freitag-{day_type}_*.Datei{table_number}_ab.csv"
    files = list(DATA_DIR.glob(pattern))
    if len(files) != 1:
        raise FileNotFoundError(f"Expected one source file for {year} {day_type}; found {files}")
    return files[0]


def frequency_table(year, day_type, departure_column):
    table = read_frequency_csv(source_file(year, day_type)).rename(columns={
        "EFA-Haltestellennummer": "efa_stop_id",
        "GlobalID": "global_id",
        "Haltestellenname mit Ort": "station_name",
        "Anzahl Abfahrten": departure_column,
    })
    table["efa_stop_id"] = pd.to_numeric(table["efa_stop_id"], errors="raise").astype("int64")
    table["station_id"] = table["efa_stop_id"] - EFA_PREFIX
    table[departure_column] = pd.to_numeric(table[departure_column], errors="raise")
    return table[["station_id", "efa_stop_id", "global_id", "station_name", departure_column]]


def weekday_line_table(year):
    tables = []
    for day_type in ("Schule", "Ferien"):
        lines = read_frequency_csv(source_file(year, day_type, table_number=2))
        line_key_column = next(column for column in lines.columns if column.startswith("Linienschl"))
        lines = lines.rename(columns={
            "EFA-Haltestellennummer": "efa_stop_id",
            "Linientext": "line_name",
            line_key_column: "line_key",
        })[["efa_stop_id", "line_key", "line_name"]]
        lines["efa_stop_id"] = pd.to_numeric(lines["efa_stop_id"], errors="raise").astype("int64")
        lines["station_id"] = lines["efa_stop_id"] - EFA_PREFIX
        lines["line_name"] = lines["line_name"].astype("string").str.strip()
        lines["route_id"] = lines["line_key"].astype("string").str.strip().str.replace(r"\..*$", "", regex=True)
        missing_route_id = lines["route_id"].isna() | lines["route_id"].eq("")
        lines.loc[missing_route_id, "route_id"] = "TEXT:" + lines.loc[missing_route_id, "line_name"].str.upper().str.replace(r"\s+", "", regex=True)
        tables.append(lines.dropna(subset=["line_name"]))

    station_lines = pd.concat(tables, ignore_index=True)
    station_lines = station_lines[station_lines["line_name"].ne("")]
    line_counts = (station_lines.drop_duplicates(["station_id", "line_name"])
                   .groupby("station_id", as_index=False).size()
                   .rename(columns={"size": "weekday_line_count"}))
    route_ids = (station_lines.drop_duplicates(["station_id", "route_id"])
                 .groupby("station_id")["route_id"]
                 .agg(lambda values: "|".join(sorted(values)))
                 .rename("weekday_route_ids").reset_index())
    return line_counts.merge(route_ids, on="station_id", how="outer", validate="one_to_one")


## Stop locations


In [25]:
locations = pd.read_excel(DATA_DIR / "Manfred_Brandl_Haltestellen-Mangurano.xlsx", sheet_name="Haltestellen")
locations = locations.rename(columns={
    "Nummer": "station_id",
    "Name mit Ort": "station_name_location_file",
    "MRCV-x": "x_web_mercator",
    "MRCV-y2": "y_web_mercator",
})[["station_id", "station_name_location_file", "x_web_mercator", "y_web_mercator"]]
locations["station_id"] = pd.to_numeric(locations["station_id"], errors="raise").astype("int64")
locations[["x_web_mercator", "y_web_mercator"]] = locations[["x_web_mercator", "y_web_mercator"]].apply(pd.to_numeric, errors="coerce")
locations = locations.dropna(subset=["x_web_mercator", "y_web_mercator"])

print(f"Located stops in workbook: {len(locations):,}")
display(locations.head())
display(locations[["x_web_mercator", "y_web_mercator"]].describe().round(1))


Located stops in workbook: 10,257


,station_id,station_name_location_file,x_web_mercator,y_web_mercator
0,1,Voitsberg Josefskirche,1685862,5950405
1,2,Voitsberg Hauptplatz,1686518,5950126
2,3,Voitsberg Hauptschule,1686809,5949442
3,4,Arnstein Gustibar,1686924,5948687
4,5,Arnstein Koppn Kapelle,1688678,5947287


,x_web_mercator,y_web_mercator
count,10257.0,10257.0
mean,1705591.3,5967704.2
std,65443.4,41324.6
min,1510476.0,5883162.0
25%,1686711.0,5941359.0
50%,1718828.0,5963854.0
75%,1753814.0,6003041.0
max,1796812.0,6075805.0


## Check the annual source extracts


In [26]:
raw_tables = {}
source_rows = []

for year in SOURCE_YEARS:
    school = frequency_table(year, "Schule", "weekday_school_departures")
    holiday = frequency_table(year, "Ferien", "weekday_holiday_departures")
    raw_tables[year] = {"school": school, "holiday": holiday}
    source_rows.append({
        "year": year,
        "school_stops": len(school),
        "holiday_stops": len(holiday),
        "combined_stops": len(set(school["station_id"]) | set(holiday["station_id"])),
        "school_departures": int(school["weekday_school_departures"].sum()),
        "holiday_departures": int(holiday["weekday_holiday_departures"].sum()),
    })

source_comparison = pd.DataFrame(source_rows)
source_comparison


,year,school_stops,holiday_stops,combined_stops,school_departures,holiday_departures
0,2016,2542,4808,4880,122746,200239
1,2017,7519,4854,7630,247073,203911
2,2018,7401,4673,7469,243321,191159
3,2019,7399,4663,7454,253893,214967
4,2020,7078,4744,7166,264170,212566
5,2022,7065,4666,7093,283503,227813


### Decision for 2016

The 2016 holiday extract is comparable with the surrounding years, but its school-day extract is clearly incomplete. The final 2016 layer is therefore a full copy of 2017 with the target year changed to 2016. This avoids treating missing source records as real zero service and keeps the choice visible in the data.


In [27]:
def build_year_dataset(year):
    school = raw_tables[year]["school"]
    holiday = raw_tables[year]["holiday"]

    frequencies = school.merge(
        holiday,
        on="station_id",
        how="outer",
        suffixes=("_school", "_holiday"),
        validate="one_to_one",
    )

    # A stop can appear in only one day type. Keep its identifiers and name from whichever table contains it.
    for column in ("efa_stop_id", "global_id", "station_name"):
        frequencies[column] = frequencies[f"{column}_school"].combine_first(frequencies[f"{column}_holiday"])

    frequencies["school_source_has_stop"] = frequencies["weekday_school_departures"].notna()
    frequencies["holiday_source_has_stop"] = frequencies["weekday_holiday_departures"].notna()
    frequencies[["weekday_school_departures", "weekday_holiday_departures"]] = frequencies[
        ["weekday_school_departures", "weekday_holiday_departures"]
    ].fillna(0)
    frequencies["weekday_avg_departures"] = frequencies[
        ["weekday_school_departures", "weekday_holiday_departures"]
    ].mean(axis=1)

    frequencies = frequencies.merge(weekday_line_table(year), on="station_id", how="left", validate="one_to_one")
    frequencies["weekday_line_count"] = frequencies["weekday_line_count"].fillna(0).astype("int64")
    frequencies["weekday_route_ids"] = frequencies["weekday_route_ids"].fillna("").astype("string")

    dataset = frequencies.merge(locations, on="station_id", how="left", validate="one_to_one")
    dataset["year"] = year
    dataset["frequency_source_year"] = year
    dataset["imputation_method"] = "observed"
    dataset["location_matched"] = dataset[["x_web_mercator", "y_web_mercator"]].notna().all(axis=1)
    geometry = gpd.GeoSeries(
        gpd.points_from_xy(dataset["x_web_mercator"], dataset["y_web_mercator"]),
        crs="EPSG:3857",
    )
    geometry.loc[~dataset["location_matched"]] = None
    dataset = gpd.GeoDataFrame(dataset, geometry=geometry, crs="EPSG:3857")

    columns = [
        "year", "frequency_source_year", "imputation_method",
        "station_id", "efa_stop_id", "global_id", "station_name", "station_name_location_file",
        "weekday_school_departures", "weekday_holiday_departures", "weekday_avg_departures",
        "weekday_line_count", "weekday_route_ids", "school_source_has_stop", "holiday_source_has_stop",
        "location_matched", "x_web_mercator", "y_web_mercator", "geometry",
    ]
    return dataset[columns].sort_values("station_id").reset_index(drop=True)


datasets = {year: build_year_dataset(year) for year in VALID_SOURCE_YEARS}

# Hardcoded analytical decision: 2017 is the complete proxy for the defective 2016 source year.
datasets[2016] = datasets[2017].copy()
datasets[2016]["year"] = 2016
datasets[2016]["frequency_source_year"] = 2017
datasets[2016]["imputation_method"] = "full_2017_proxy"


## Add the supplied 2025 layer


In [28]:
stops_2025 = gpd.read_file(f"zip://{(OUTPUT_DIR / 'Haltestellen_2025.zip').resolve()}").to_crs("EPSG:3857")
stops_2025["station_id"] = pd.to_numeric(stops_2025["HNR"], errors="raise").astype("int64")
stops_2025["efa_stop_id"] = stops_2025["station_id"] + EFA_PREFIX
stops_2025["global_id"] = "at:46:" + stops_2025["station_id"].astype("string")
stops_2025["station_name"] = stops_2025["HNAME_LANG"].astype("string")
stops_2025["station_name_location_file"] = stops_2025["station_name"]
stops_2025["weekday_school_departures"] = pd.to_numeric(stops_2025["MoFr_S"], errors="raise")
stops_2025["weekday_holiday_departures"] = pd.to_numeric(stops_2025["MoFr_F"], errors="raise")
stops_2025["weekday_avg_departures"] = stops_2025[["weekday_school_departures", "weekday_holiday_departures"]].mean(axis=1)
stops_2025["weekday_line_count"] = stops_2025["LINIE"].fillna("").map(
    lambda value: len({line.strip() for line in str(value).split(",") if line.strip()})
).astype("int64")
stops_2025["weekday_route_ids"] = stops_2025["LINIE"].fillna("").map(
    lambda value: "|".join(sorted({line.strip() for line in str(value).split(",") if line.strip()}))
).astype("string")
stops_2025["school_source_has_stop"] = True
stops_2025["holiday_source_has_stop"] = True
stops_2025["location_matched"] = stops_2025.geometry.notna() & ~stops_2025.geometry.is_empty
stops_2025["x_web_mercator"] = stops_2025.geometry.x
stops_2025["y_web_mercator"] = stops_2025.geometry.y
stops_2025["year"] = 2025
stops_2025["frequency_source_year"] = 2025
stops_2025["imputation_method"] = "observed"

datasets[2025] = stops_2025[datasets[2017].columns].sort_values("station_id").reset_index(drop=True)


## Inspect and export the final annual layers


In [29]:
validation = pd.DataFrame([
    {
        "year": year,
        "frequency_source_year": int(dataset["frequency_source_year"].iloc[0]),
        "imputation_method": dataset["imputation_method"].iloc[0],
        "frequency_stops": len(dataset),
        "coordinate_matches": int(dataset["location_matched"].sum()),
        "unmatched_coordinates": int((~dataset["location_matched"]).sum()),
        "missing_station_names": int(dataset["station_name"].isna().sum()),
    }
    for year, dataset in sorted(datasets.items())
])

summary = pd.DataFrame([
    {
        "year": year,
        "stops": len(dataset),
        "located_stops": int(dataset["location_matched"].sum()),
        "total_avg_departures": dataset["weekday_avg_departures"].sum(),
        "median_avg_departures": dataset["weekday_avg_departures"].median(),
        "median_weekday_line_count": dataset["weekday_line_count"].median(),
        "max_weekday_line_count": dataset["weekday_line_count"].max(),
    }
    for year, dataset in sorted(datasets.items())
])

for year, dataset in datasets.items():
    dataset.to_parquet(OUTPUT_DIR / f"public_transport_weekday_stop_frequency_{year}.geoparquet", index=False)

validation.to_csv(OUTPUT_DIR / "public_transport_weekday_frequency_validation.csv", index=False)
summary.to_csv(OUTPUT_DIR / "public_transport_weekday_frequency_summary.csv", index=False)

comparison_columns = [column for column in datasets[2017].columns if column not in {"year", "frequency_source_year", "imputation_method"}]
pd.testing.assert_frame_equal(datasets[2016][comparison_columns], datasets[2017][comparison_columns])

display(validation)
display(summary)
display(datasets[2016].head())
display(datasets[2016][["weekday_school_departures", "weekday_holiday_departures", "weekday_avg_departures", "weekday_line_count"]].describe().round(2))


,year,frequency_source_year,imputation_method,frequency_stops,coordinate_matches,unmatched_coordinates,missing_station_names
0,2016,2017,full_2017_proxy,7630,7562,68,0
1,2017,2017,observed,7630,7562,68,0
2,2018,2018,observed,7469,7444,25,0
3,2019,2019,observed,7454,7426,28,0
4,2020,2020,observed,7166,7117,49,0
5,2022,2022,observed,7093,7081,12,0
6,2025,2025,observed,8221,8221,0,0


,year,stops,located_stops,total_avg_departures,median_avg_departures,median_weekday_line_count,max_weekday_line_count
0,2016,7630,7562,225492.0,7.0,1.0,67
1,2017,7630,7562,225492.0,7.0,1.0,67
2,2018,7469,7444,217240.0,7.0,1.0,57
3,2019,7454,7426,234430.0,7.0,1.0,59
4,2020,7166,7117,238368.0,8.5,1.0,60
5,2022,7093,7081,255658.0,9.5,1.0,51
6,2025,8221,8221,255591.5,6.0,1.0,37


,year,frequency_source_year,imputation_method,station_id,efa_stop_id,global_id,station_name,station_name_location_file,weekday_school_departures,weekday_holiday_departures,weekday_avg_departures,weekday_line_count,weekday_route_ids,school_source_has_stop,holiday_source_has_stop,location_matched,x_web_mercator,y_web_mercator,geometry
0,2016,2017,full_2017_proxy,1,63200001.0,at:46:1,Voitsberg Josefskirche,Voitsberg Josefskirche,60.0,48.0,54.0,4,86700_|86702_|86705_|86710_,True,True,True,1685862.0,5950405.0,POINT (1685862 5950405)
1,2016,2017,full_2017_proxy,2,63200002.0,at:46:2,Voitsberg Hauptplatz,Voitsberg Hauptplatz,91.0,63.0,77.0,7,09713_|09718_|86700_|86702_|86705_|86710_|86727a,True,True,True,1686518.0,5950126.0,POINT (1686518 5950126)
2,2016,2017,full_2017_proxy,4,63200004.0,at:46:4,Arnstein Gustibar,Arnstein Gustibar,4.0,0.0,2.0,1,01717_,True,False,True,1686924.0,5948687.0,POINT (1686924 5948687)
3,2016,2017,full_2017_proxy,5,63200005.0,at:46:5,Arnstein Koppn Kapelle,Arnstein Koppn Kapelle,5.0,0.0,2.5,1,01717_,True,False,True,1688678.0,5947287.0,POINT (1688678 5947287)
4,2016,2017,full_2017_proxy,8,63200008.0,at:46:8,Arnstein ehemaliges Gemeindeamt,Arnstein ehemaliges Gemeindeamt,5.0,0.0,2.5,1,01717_,True,False,True,1688745.0,5946436.0,POINT (1688745 5946436)


,weekday_school_departures,weekday_holiday_departures,weekday_avg_departures,weekday_line_count
count,7630.00,7630.00,7630.00,7630.00
mean,32.38,26.72,29.55,1.84
std,87.78,81.57,83.40,2.31
min,0.00,0.00,0.50,1.00
25%,4.00,0.00,2.50,1.00
50%,9.00,4.00,7.00,1.00
75%,25.00,18.00,21.00,2.00
max,2588.00,2250.00,2419.00,67.00


## Interactive station map for one year

Set `MAP_YEAR` to one of the generated source years. The Leaflet map contains every station with valid coordinates and shows its weekday departures and direction-deduplicated line count. The resulting HTML file is standalone apart from the web map tiles.

In [30]:
import json

import folium

MAP_YEAR = 2025
MAP_OUTPUT_DIR = Path("FIG/Haltestellen")
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

map_path = OUTPUT_DIR / f"public_transport_weekday_stop_frequency_{MAP_YEAR}.geoparquet"
map_stops = gpd.read_parquet(map_path).dropna(subset=["geometry"]).to_crs("EPSG:4326")
map_stops = map_stops[["station_id", "station_name", "weekday_avg_departures", "weekday_line_count", "geometry"]].copy()
map_center = [map_stops.geometry.y.median(), map_stops.geometry.x.median()]

station_map = folium.Map(location=map_center, zoom_start=8, tiles="CartoDB positron", control_scale=True)
folium.GeoJson(
    data=json.loads(map_stops.to_json()),
    name=f"Public-transport stations {MAP_YEAR}",
    marker=folium.CircleMarker(radius=3, color="#1f4e79", weight=1, fill=True, fill_opacity=0.75),
    tooltip=folium.GeoJsonTooltip(
        fields=["station_name", "weekday_line_count", "weekday_avg_departures"],
        aliases=["Station:", "Weekday lines:", "Average weekday departures:"],
        localize=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=["station_id", "station_name", "weekday_line_count", "weekday_avg_departures"],
        aliases=["Station ID:", "Station:", "Weekday lines:", "Average weekday departures:"],
        localize=True,
    ),
).add_to(station_map)
folium.LayerControl().add_to(station_map)

map_output = MAP_OUTPUT_DIR / f"public_transport_stations_{MAP_YEAR}_interactive.html"
station_map.save(map_output)
print(f"Mapped {len(map_stops):,} stations and wrote {map_output}")


Mapped 8,221 stations and wrote FIG\Haltestellen\public_transport_stations_2025_interactive.html
